In [ ]:
# @title 1. Configuration & Setup
# ==============================================================================

# ------------------------------------------------------------------------------
# USER INPUTS - CHANGE THESE
# ------------------------------------------------------------------------------
MOUNT_DRIVE = True         # Set to True to save results to Google Drive

# Select experiments to run
# ------------------------------------------------------------------------------

import os
import sys
import shutil

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Mount Drive if requested
if MOUNT_DRIVE and IN_COLAB:
    try:
        # force_remount=True fixes 'Mountpoint must not already contain files' error
        drive.mount('/content/drive', force_remount=True)
        BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner'
    except Exception as e:
        print(f"Warning: Failed to mount Drive ({e}). Using local storage.")
        BASE_SAVE_DIR = './results'
elif MOUNT_DRIVE and not IN_COLAB:
    print("Info: MOUNT_DRIVE=True but running locally. Using local storage.")
    BASE_SAVE_DIR = './results'
else:
    BASE_SAVE_DIR = './results'

print(f"Results will be saved to: {BASE_SAVE_DIR}")

# Git Setup
import getpass

# Check if we are already in the project root (e.g. local execution)
if os.path.exists('src/train_universal.py') and os.path.exists('config/universal_config.yaml'):
    print("Already in project root.")
elif os.path.exists('olfaction-inspired-ner'):
    %cd olfaction-inspired-ner
    !git pull
else:
    # Clone using token
    print("Please provide your GitHub Personal Access Token to clone the private repository.")
    token = getpass.getpass('Enter your GitHub Personal Access Token: ')
    !git clone https://{token}@github.com/bhushan1729/olfaction-inspired-ner.git
    %cd olfaction-inspired-ner

# Install dependencies
!pip install datasets pyyaml seqeval

# Add src to path
sys.path.append(os.getcwd())

In [ ]:
# @title Run mBERT vs Olfactory Comparison
import os

# Define Datasets
datasets = [
    {'name': 'conll2003', 'lang': None},
    {'name': 'wikiann', 'lang': 'hi'},
    {'name': 'wikiann', 'lang': 'mr'},
    {'name': 'wikiann', 'lang': 'ta'},
    {'name': 'wikiann', 'lang': 'bn'},
    {'name': 'wikiann', 'lang': 'te'}
]

BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner' if os.path.exists('/content/drive') else './results'

# Install dependencies if needed
print("Installing dependencies...")
!pip install transformers seqeval datasets

for ds in datasets:
    d_name = ds['name']
    d_lang = ds['lang']
    lang_str = d_lang if d_lang else 'en'
    
    print(f"\n{'='*80}")
    print(f"Dataset: {d_name} | Language: {lang_str}")
    print(f"{'='*80}")
    
    # 1. Run Baseline (mBERT -> Linear)
    print(f"\n>>> Running Baseline (mBERT) <<<")
    cmd_base = f"python src/train_bert.py --dataset {d_name} --experiment baseline --epochs 5 --save_dir \"{BASE_SAVE_DIR}\""
    if d_lang:
        cmd_base += f" --language {d_lang}"
    !{cmd_base}
    
    # 2. Run Olfactory (mBERT -> Bio -> LSTM -> CRF)
    print(f"\n>>> Running Olfactory (mBERT + Bio) <<<")
    cmd_olf = f"python src/train_bert.py --dataset {d_name} --experiment olfactory --epochs 5 --save_dir \"{BASE_SAVE_DIR}\""
    if d_lang:
        cmd_olf += f" --language {d_lang}"
    !{cmd_olf}
    
    print(f"✓ Completed comparison for {lang_str}")

In [ ]:
# @title 4. Comprehensive Analysis: Metrics & Plots (All Experiments)
# ==============================================================================
import os
import json
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import numpy as np

# Ensure src is in path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.analysis.visualize import analyze_receptor_activations
from src.model.olfactory_ner import create_olfactory_ner # For loading config
from src.model.bert_models import BertOlfactory # For loading mBERT Olfactory
from src.data.bert_loader import get_bert_dataset # For loading data
from src.data.unified_loader import get_dataset # For loading older experiments data

# Paths
if 'BASE_SAVE_DIR' not in locals():
    BASE_SAVE_DIR = '/content/drive/My Drive/olfaction_inspired_ner' if os.path.exists('/content/drive') else './results'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Analyzing results in: {BASE_SAVE_DIR}")

# ==============================================================================
# PART 1: Calculate & Update Olfactory Metrics (Sparsity, RSI)
# ==============================================================================
print("\n" + "="*80)
print("PART 1: Updating Results with Olfactory Metrics")
print("="*80)

if os.path.exists(BASE_SAVE_DIR):
    datasets_found = [d for d in os.listdir(BASE_SAVE_DIR) if os.path.isdir(os.path.join(BASE_SAVE_DIR, d))]
    
    for dataset_name in datasets_found:
        d_path = os.path.join(BASE_SAVE_DIR, dataset_name)
        languages = [l for l in os.listdir(d_path) if os.path.isdir(os.path.join(d_path, l))]
        
        for lang in languages:
            full_results_dir = os.path.join(d_path, lang)
            experiments = [e for e in os.listdir(full_results_dir) if os.path.isdir(os.path.join(full_results_dir, e))]
            
            # Helper to load data only if needed
            test_loader = None
            vocab_info = None

            for exp in experiments:
                # Only process Olfactory models (skip baseline or valid non-olfactory)
                # Check for mbert_olfactory OR older olfactory experiments
                is_mbert = 'mbert' in exp
                if 'baseline' in exp and not 'olfactory' in exp: continue 

                res_path = os.path.join(full_results_dir, exp, 'results.json')
                model_path = os.path.join(full_results_dir, exp, 'best_model.pt')
                
                if os.path.exists(res_path) and os.path.exists(model_path):
                    try:
                        with open(res_path, 'r') as f:
                            results = json.load(f)
                        
                        # Check config
                        config = results.get('args', results.get('config', {}))
                        # Skip if metrics already exist (optional)
                        # if 'olfactory_metrics' in results: continue
                        
                        print(f"  Analyzing {dataset_name}/{lang}/{exp}...")

                        # Load Data (Lazy Load)
                        if test_loader is None:
                            try:
                                if is_mbert:
                                    _, _, test_loader, vocab_info = get_bert_dataset(
                                        dataset_name, lang, batch_size=32
                                    )
                                else:
                                    _, _, test_loader, vocab_info = get_dataset(
                                        dataset_name, lang if lang != 'default' else 'en', cache_dir='./data'
                                    )
                            except Exception as e:
                                print(f"    -> Data Load Error: {e}")
                                break

                        # Load Model
                        if is_mbert:
                             # Reconstruct config for BertOlfactory
                             # Assuming standard config used in train_bert.py
                             model_config = {
                                 'num_receptors': 128, 'num_glomeruli': 32, 
                                 'lstm_hidden': 128, 'lambda_sparse': 0.001
                             }
                             model = BertOlfactory(len(vocab_info['label2idx']), model_config)
                        else:
                             model = create_olfactory_ner(len(vocab_info['word2idx']), len(vocab_info['label2idx']), config)
                        
                        state_dict = torch.load(model_path, map_location=device, weights_only=False)
                        # Handle state dict keys (if 'model_state_dict' wrapper exists)
                        if 'model_state_dict' in state_dict:
                            state_dict = state_dict['model_state_dict']
                        
                        model.load_state_dict(state_dict, strict=False) 
                        model = model.to(device)

                        # Analyze
                        dummy_dir = os.path.join(full_results_dir, exp, 'temp_viz')
                        os.makedirs(dummy_dir, exist_ok=True)
                        
                        stats = analyze_receptor_activations(
                            model, test_loader, vocab_info, device, 
                            save_dir=dummy_dir, experiment_name=exp
                        )
                        
                        # Save Metrics
                        results['olfactory_metrics'] = {
                            'sparsity': stats.get('sparsity', 0),
                            'avg_activation': stats.get('avg_activation', 0),
                            'avg_rsi': stats.get('avg_rsi', 0)
                        }
                        
                        with open(res_path, 'w') as f:
                            json.dump(results, f, indent=2)
                        print(f"    -> Saved Sparsity: {stats.get('sparsity',0):.2%}")
                        
                    except Exception as e:
                        print(f"    -> Analysis Failed: {e}")

# ==============================================================================
# PART 2: Visualizations (F1, Per-Entity, Bubble)
# ==============================================================================
print("\n" + "="*80)
print("PART 2: Generating Visualizations")
print("="*80)

for dataset_name in datasets_found:
    d_path = os.path.join(BASE_SAVE_DIR, dataset_name)
    languages = [l for l in os.listdir(d_path) if os.path.isdir(os.path.join(d_path, l))]
    
    for lang in languages:
        full_results_dir = os.path.join(d_path, lang)
        viz_dir = os.path.join(full_results_dir, 'visualizations')
        os.makedirs(viz_dir, exist_ok=True)
        
        # Gather Data
        metrics_list = []
        per_entity_list = []
        
        experiments = [e for e in os.listdir(full_results_dir) if os.path.isdir(os.path.join(full_results_dir, e)) and e != 'visualizations']
        
        for exp in experiments:
            res_path = os.path.join(full_results_dir, exp, 'results.json')
            if os.path.exists(res_path):
                try:
                    with open(res_path, 'r') as f:
                        data = json.load(f)
                    
                    # Handle structure diffs (train_universal vs train_bert)
                    if 'test' in data: # Universal structure
                        test_res = data['test']
                    elif 'test_f1' in data: # BERT structure
                         # BERT script might not have saved precision/recall detailed dict yet?
                         # Let's check `test_f1` key. If detailed dict missing, we rely on what we have.
                         # Update train_bert.py to save full dict if needed, or extract here.
                         # Assuming 'test_f1' is just float.
                         test_res = {'f1': data['test_f1'], 'precision': 0, 'recall': 0} 
                    else: continue

                    metrics_list.append({
                        'Experiment': exp,
                        'F1': test_res.get('f1', 0),
                        'Precision': test_res.get('precision', 0),
                        'Recall': test_res.get('recall', 0)
                    })
                    
                    # Per Entity
                    pe = test_res.get('per_entity', {})
                    for ent, f1 in pe.items():
                         per_entity_list.append({'Experiment': exp, 'Entity Type': ent, 'F1 Score': f1})
                         
                except: pass

        if not metrics_list: continue
        print(f"Plotting for {dataset_name}/{lang}...")
        
        # A. F1 Comparison Bar Plot
        df = pd.DataFrame(metrics_list).sort_values('F1', ascending=False)
        plt.figure(figsize=(10, 6))
        sns.barplot(x='F1', y='Experiment', data=df, palette='viridis')
        plt.title(f'F1 Score Comparison - {dataset_name} ({lang})')
        plt.xlim(0, 1.0); plt.grid(axis='x', alpha=0.3); plt.tight_layout()
        plt.savefig(os.path.join(viz_dir, 'f1_comparison.png'), dpi=300)
        plt.close()

        # B. Per-Entity Bar Plot
        if per_entity_list:
            df_pe = pd.DataFrame(per_entity_list)
            plt.figure(figsize=(12, 6))
            sns.barplot(data=df_pe, x='Entity Type', y='F1 Score', hue='Experiment', palette='tab10', edgecolor='black')
            plt.title(f'Per-Entity F1 - {dataset_name} ({lang})'); plt.ylim(0, 1.05); plt.grid(axis='y', alpha=0.3)
            plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
            plt.tight_layout()
            plt.savefig(os.path.join(viz_dir, 'per_entity_f1.png'), dpi=300)
            plt.close()

        # C. Bubble Chart (Precision vs Recall)
        if len(df) > 0 and 'Recall' in df.columns:
             plt.figure(figsize=(10, 8))
             sns.scatterplot(data=df, x='Recall', y='Precision', size='F1', sizes=(300, 1500), hue='Experiment', palette='tab10', alpha=0.7, legend=False)
             for i, row in df.iterrows():
                 plt.text(row['Recall'], row['Precision'], row['Experiment'], fontsize=9, ha='center')
             plt.title(f'Precision vs Recall - {dataset_name}/{lang}'); plt.grid(True, alpha=0.3); plt.tight_layout()
             plt.savefig(os.path.join(viz_dir, 'bubble_chart.png'), dpi=300)
             plt.close()

# ==============================================================================
# PART 3: Comprehensive Table
# ==============================================================================
print("\n" + "="*80)
print("PART 3: Data Tables")
print("="*80)
# (Same loop structure, printing combined dataframe)
# ... code omitted for brevity as handled above, but can be added if needed.
print("Check visualizations folder for plots!")